# MediScan -- Exploratory Data Analysis (Day 1)

> **[DISCLAIMER]:** This project is for educational/research purposes only and is not a clinical diagnostic system.

## Objective
Inspect and verify the HAM10000 dataset before building the ML pipeline.

## Sections
1. Setup & Imports
2. Dataset Loading
3. Metadata Inspection
4. Class Distribution
5. Lesion ID Analysis (Data Leakage Prevention)
6. Image Integrity Verification
7. Image Dimensions
8. Missing Values
9. Metadata Distributions (Age, Sex, Localization)
10. Sample Image Visualization
11. Observations & Data Quality Issues
12. Next Steps

## 1. Setup & Imports

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Project root -- adjust if running from a different location
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.preprocessing import (
    load_metadata, analyze_class_distribution, check_image_integrity,
    analyze_duplicates, analyze_metadata_quality, find_metadata_csv,
    find_image_directories,
)
from src.utils.seed import set_seed

set_seed(42)

# Display settings
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {sys.version}')

## 2. Dataset Loading

Locate and load the HAM10000 metadata CSV.

In [ ]:
# Locate metadata CSV
csv_path = find_metadata_csv(str(PROJECT_ROOT / 'data' / 'metadata'))
if csv_path is None:
    csv_path = find_metadata_csv(str(PROJECT_ROOT / 'data' / 'raw'))
if csv_path is None:
    csv_path = find_metadata_csv(str(PROJECT_ROOT / 'data'))

if csv_path is None:
    raise FileNotFoundError(
        'Metadata CSV not found! '
        'Run `python scripts/download_dataset.py` or manually place '
        'HAM10000_metadata.csv in data/raw/'
    )

print(f'Metadata CSV: {csv_path}')

# Locate image directories
image_dirs = find_image_directories(str(PROJECT_ROOT / 'data' / 'raw'))
print(f'Image directories: {image_dirs}')

## 3. Metadata Inspection

In [ ]:
df = load_metadata(csv_path)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print()
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 4. Class Distribution

In [ ]:
CLASS_NAMES = {
    'akiec': 'Actinic Keratoses',
    'bcc': 'Basal Cell Carcinoma',
    'bkl': 'Benign Keratosis',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic Nevi',
    'vasc': 'Vascular Lesions',
}

dist = analyze_class_distribution(df)
print(f"Number of classes: {dist['num_classes']}")
print(f"Class labels: {dist['class_labels']}")
print()
for label in sorted(dist['class_counts'].keys()):
    name = CLASS_NAMES.get(label, label)
    count = dist['class_counts'][label]
    pct = dist['class_percentages'][label]
    print(f'  {label:>6} ({name:>25}): {count:>6}  ({pct:>5}%)')

In [ ]:
# Class distribution bar chart
counts = dist['class_counts']
labels = sorted(counts.keys())
values = [counts[l] for l in labels]
full_names = [f"{l}\n({CLASS_NAMES.get(l, l)})" for l in labels]

fig, ax = plt.subplots(figsize=(12, 6))
colors = sns.color_palette('viridis', len(labels))
bars = ax.bar(full_names, values, color=colors, edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, values):
    pct = val / sum(values) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)

ax.set_title('HAM10000 -- Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Diagnosis Class', fontsize=11)
ax.set_ylabel('Number of Images', fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Lesion ID Analysis (Data Leakage Prevention)

HAM10000 contains multiple images for some lesions. If images from the same lesion
end up in different splits, the model can memorize lesion-specific features,
causing **data leakage** and inflated test metrics.

**The train/val/test split MUST be performed at the `lesion_id` level.**

In [ ]:
dup_stats = analyze_duplicates(df)

if 'error' not in dup_stats:
    print(f"Unique lesions: {dup_stats['unique_lesions']}")
    print(f"Total images: {dup_stats['total_images']}")
    print(f"Multi-image lesions: {dup_stats['multi_image_lesions']}")
    print(f"Single-image lesions: {dup_stats['single_image_lesions']}")
    ipl = dup_stats['images_per_lesion']
    print(f"Images per lesion -- min: {ipl['min']}, max: {ipl['max']}, "
          f"mean: {ipl['mean']}, median: {ipl['median']}")
    print()
    print(f"[WARNING] LEAKAGE RISK: {dup_stats['leakage_risk']}")
else:
    print(dup_stats['error'])

In [ ]:
# Images-per-lesion distribution
if 'distribution' in dup_stats:
    dist_data = dup_stats['distribution']
    keys = sorted(dist_data.keys())
    vals = [dist_data[k] for k in keys]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar([str(k) for k in keys], vals, color=sns.color_palette('muted')[0],
           edgecolor='black', linewidth=0.5)
    ax.set_title('Distribution of Images per Lesion', fontsize=14, fontweight='bold')
    ax.set_xlabel('Images per Lesion')
    ax.set_ylabel('Number of Lesions')
    ax.grid(axis='y', alpha=0.3)
    for i, (k, v) in enumerate(zip(keys, vals)):
        ax.text(i, v + max(vals)*0.01, str(v), ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()

## 6. Image Integrity Verification

In [ ]:
if image_dirs:
    image_ids = df['image_id'].tolist() if 'image_id' in df.columns else None
    integrity = check_image_integrity(image_dirs[0], image_ids=image_ids, sample_size=500)
    
    print(f"Files found in first directory: {integrity['total_files']}")
    print(f"Inspected: {integrity['inspected_count']}")
    print(f"Readable: {integrity['readable']}")
    print(f"Corrupted: {len(integrity['corrupted'])}")
    print(f"Formats: {integrity['formats']}")
    print(f"Modes: {integrity['modes']}")
    print(f"Dimensions: {integrity['dimensions']}")
    print(f"Missing from disk: {len(integrity['missing_from_dir'])}")
    print(f"Orphan files: {len(integrity['orphan_files'])}")
else:
    print('No image directories found.')

## 7. Image Dimensions

Check actual image dimensions. All will be resized to 224x224 for model input.

In [ ]:
if image_dirs:
    print('Observed image dimensions (from sample):')
    for dim, count in integrity.get('dimensions', {}).items():
        print(f'  {dim}: {count} images')
    print()
    print('All images will be resized to 224x224 for model input (Day 2).')

## 8. Missing Values

In [ ]:
meta_quality = analyze_metadata_quality(df)

print('Missing values per column:')
for col, count in meta_quality['missing_values'].items():
    pct = meta_quality['missing_percentages'][col]
    status = '[OK] complete' if count == 0 else f'[WARNING] {count} missing ({pct}%)'
    print(f'  {col:>15}: {status}')

## 9. Metadata Distributions

Contextual analysis for later bias/edge-case investigation (Day 14).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age
if 'age' in df.columns and df['age'].notna().sum() > 0:
    df['age'].dropna().hist(bins=20, ax=axes[0], color=sns.color_palette('muted')[0],
                            edgecolor='black')
    axes[0].set_title('Age Distribution', fontweight='bold')
    axes[0].set_xlabel('Age')

# Sex
if 'sex' in df.columns and df['sex'].notna().sum() > 0:
    sex_counts = df['sex'].value_counts()
    axes[1].bar(sex_counts.index.astype(str), sex_counts.values,
                color=sns.color_palette('muted')[1:len(sex_counts)+1], edgecolor='black')
    axes[1].set_title('Sex Distribution', fontweight='bold')

# Localization
if 'localization' in df.columns and df['localization'].notna().sum() > 0:
    loc_counts = df['localization'].value_counts().head(10)
    axes[2].barh(loc_counts.index[::-1], loc_counts.values[::-1],
                 color=sns.color_palette('muted')[2], edgecolor='black')
    axes[2].set_title('Top 10 Localizations', fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Sample Image Visualization

Display representative samples from each class.

In [ ]:
if image_dirs:
    classes = sorted(df['dx'].unique())
    samples_per_class = 3
    
    fig, axes = plt.subplots(len(classes), samples_per_class,
                             figsize=(4*samples_per_class, 3.5*len(classes)))
    
    for i, cls in enumerate(classes):
        cls_df = df[df['dx'] == cls].sample(
            n=min(samples_per_class, len(df[df['dx'] == cls])), random_state=42
        )
        for j, (_, row) in enumerate(cls_df.iterrows()):
            img_path = None
            for d in image_dirs:
                for ext in ['.jpg', '.jpeg', '.png']:
                    candidate = Path(d) / f"{row['image_id']}{ext}"
                    if candidate.exists():
                        img_path = candidate
                        break
                if img_path:
                    break
            
            ax = axes[i, j]
            if img_path:
                img = Image.open(img_path)
                ax.imshow(img)
                ax.set_title(f"{cls} ({CLASS_NAMES.get(cls, cls)})", fontsize=9)
            else:
                ax.text(0.5, 0.5, f'Not found: {row["image_id"]}',
                        ha='center', va='center')
            ax.axis('off')
    
    plt.suptitle('Sample Images per Class', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print('No image directories found.')

## 11. Observations & Data Quality Issues

### Key Findings

### Class Imbalance
- HAM10000 is **heavily imbalanced** -- the `nv` (Melanocytic Nevi) class dominates at 66.95%
- Minority classes (`df`: 1.15%, `vasc`: 1.42%) have very few samples
- This must be addressed during training (Day 6+) with weighted loss and/or sampling strategies

### Data Leakage Risk
- Multiple images exist per lesion via `lesion_id` (1956 multi-image lesions)
- **The train/val/test split MUST group by `lesion_id`** to prevent data leakage
- A simple random image-level split would be INVALID

### Ethical Considerations
- This dataset has **known demographic limitations**
- It does not contain explicit skin tone labels -- this is a **dataset limitation**, not something we can measure
- The model should never be used for clinical diagnosis

## 12. Next Steps (Day 2)

1. Implement lesion_id-aware stratified train/val/test split (70/15/15)
2. Verify no lesion_id appears in multiple splits
3. Resize images to 224x224
4. Apply ImageNet normalization
5. Report split class distributions